In [ ]:
# ==========================================
# 0. УСТАНОВКА ЗАВИСИМОСТЕЙ
# ==========================================
!pip install django Pillow

import os
import time
import threading
from google.colab.output import eval_js

# ==========================================
# 1. СОЗДАНИЕ ПРОЕКТА
# ==========================================
%cd /content
!django-admin startproject config
%cd /content/config
!python manage.py startapp storage

# ==========================================
# 2. НАСТРОЙКА SETTINGS.PY
# ==========================================
filepath = '/content/config/config/settings.py'
with open(filepath, 'r') as f:
    code = f.read()

code = code.replace(
    "'django.contrib.staticfiles',",
    "'django.contrib.staticfiles',\n    'storage',"
)

code += """
import os
STATIC_URL = '/static/'
STATICFILES_DIRS = [os.path.join(BASE_DIR, 'storage/static')]
MEDIA_URL = '/media/'
MEDIA_ROOT = os.path.join(BASE_DIR, 'media')

LANGUAGE_CODE = 'ru-ru'
TIME_ZONE = 'Europe/Moscow'
USE_TZ = True

LOGIN_URL = 'login'
LOGIN_REDIRECT_URL = 'file_list'
LOGOUT_REDIRECT_URL = 'login'

# Для работы в Colab
ALLOWED_HOSTS = ['*']
CSRF_TRUSTED_ORIGINS = ['https://*.colab.dev']
SECURE_PROXY_SSL_HEADER = ('HTTP_X_FORWARDED_PROTO', 'https')
"""
with open(filepath, 'w') as f:
    f.write(code)

# ==========================================
# 3. МОДЕЛИ И АДМИНКА
# ==========================================
models_code = """
from django.db import models
from django.contrib.auth.models import User
from django.urls import reverse

class UserFile(models.Model):
    FILE_TYPES = [
        ('image', 'Изображение'),
        ('video', 'Видео'),
        ('document', 'Документ'),
        ('other', 'Другое'),
    ]
    owner = models.ForeignKey(User, on_delete=models.CASCADE, related_name='files', verbose_name="Владелец")
    file = models.FileField(upload_to='uploads/%Y/%m/%d/', verbose_name="Файл")
    title = models.CharField(max_length=200, blank=True, verbose_name="Название")
    file_type = models.CharField(max_length=20, choices=FILE_TYPES, default='other', verbose_name="Тип файла")
    size = models.PositiveBigIntegerField(default=0, verbose_name="Размер, байт")
    is_public = models.BooleanField(default=False, verbose_name="Публичный")
    uploaded_at = models.DateTimeField(auto_now_add=True, verbose_name="Загружен")

    class Meta:
        verbose_name = "Файл"
        verbose_name_plural = "Файлы"
        ordering = ['-uploaded_at']

    def __str__(self):
        return self.title or self.file.name

    def get_absolute_url(self):
        return reverse('file_detail', kwargs={'pk': self.pk})

    @property
    def is_image(self): return self.file_type == 'image'
    @property
    def is_video(self): return self.file_type == 'video'

    def save(self, *args, **kwargs):
        if self.file and not self.size:
            self.size = self.file.size
        if self.file and not self.file_type or self.file_type == 'other':
            name = self.file.name.lower()
            if name.endswith(('.jpg', '.jpeg', '.png', '.gif', '.webp')): self.file_type = 'image'
            elif name.endswith(('.mp4', '.avi', '.mov', '.mkv', '.webm')): self.file_type = 'video'
            elif name.endswith(('.pdf', '.doc', '.docx', '.txt', '.xlsx')): self.file_type = 'document'
        super().save(*args, **kwargs)

    def size_display(self):
        size = self.size
        for unit in ['Б', 'КБ', 'МБ', 'ГБ']:
            if size < 1024: return f"{size:.1f} {unit}"
            size /= 1024
        return f"{size:.1f} ТБ"
"""
admin_code = """
from django.contrib import admin
from .models import UserFile

@admin.register(UserFile)
class UserFileAdmin(admin.ModelAdmin):
    list_display = ['title', 'owner', 'file_type', 'size_display', 'is_public', 'uploaded_at']
    list_filter = ['file_type', 'is_public', 'uploaded_at']
    search_fields = ['title', 'file', 'owner__username']
"""
with open('/content/config/storage/models.py', 'w') as f: f.write(models_code)
with open('/content/config/storage/admin.py', 'w') as f: f.write(admin_code)

# ==========================================
# 4. ФОРМЫ
# ==========================================
forms_code = """
from django import forms
from django.contrib.auth.forms import UserCreationForm
from django.contrib.auth.models import User
from .models import UserFile

class RegisterForm(UserCreationForm):
    email = forms.EmailField(required=True, label="Email")
    class Meta:
        model = User
        fields = ['username', 'email', 'password1', 'password2']

class FileUploadForm(forms.ModelForm):
    class Meta:
        model = UserFile
        fields = ['file', 'title', 'is_public']
        widgets = {'title': forms.TextInput(attrs={'placeholder': 'Название (необязательно)'})}
    def clean_file(self):
        file = self.cleaned_data['file']
        if file.size > 100 * 1024 * 1024: raise forms.ValidationError("Файл слишком большой (максимум 100 МБ).")
        allowed = ('.jpg', '.jpeg', '.png', '.gif', '.webp', '.mp4', '.avi', '.mov', '.mkv', '.webm', '.pdf', '.doc', '.docx', '.txt', '.xlsx')
        if not file.name.lower().endswith(allowed): raise forms.ValidationError("Недопустимый тип файла.")
        return file
"""
with open('/content/config/storage/forms.py', 'w') as f: f.write(forms_code)

# ==========================================
# 5. ПРЕДСТАВЛЕНИЯ И МАРШРУТЫ
# ==========================================
views_code = """
from django.shortcuts import render, redirect, get_object_or_404
from django.contrib.auth import login
from django.contrib.auth.decorators import login_required
from django.http import FileResponse, Http404
from django.db.models import Q
from .models import UserFile
from .forms import RegisterForm, FileUploadForm

def register(request):
    if request.method == 'POST':
        form = RegisterForm(request.POST)
        if form.is_valid():
            user = form.save()
            login(request, user)
            return redirect('file_list')
    else:
        form = RegisterForm()
    return render(request, 'storage/register.html', {'form': form})

@login_required
def file_list(request):
    files = UserFile.objects.filter(owner=request.user)
    query = request.GET.get('q')
    if query: files = files.filter(Q(title__icontains=query) | Q(file__icontains=query))
    file_type = request.GET.get('type')
    if file_type in ('image', 'video', 'document', 'other'): files = files.filter(file_type=file_type)
    return render(request, 'storage/file_list.html', {'files': files, 'query': query, 'current_type': file_type})

@login_required
def upload(request):
    if request.method == 'POST':
        form = FileUploadForm(request.POST, request.FILES)
        if form.is_valid():
            obj = form.save(commit=False)
            obj.owner = request.user
            obj.save()
            return redirect('file_list')
    else:
        form = FileUploadForm()
    return render(request, 'storage/upload.html', {'form': form})

@login_required
def file_detail(request, pk):
    file = get_object_or_404(UserFile, pk=pk)
    if file.owner != request.user and not file.is_public: raise Http404("Файл не найден")
    return render(request, 'storage/file_detail.html', {'file': file})

@login_required
def file_download(request, pk):
    file = get_object_or_404(UserFile, pk=pk)
    if file.owner != request.user and not file.is_public: raise Http404("Файл не найден")
    return FileResponse(file.file.open('rb'), as_attachment=True, filename=file.file.name.split('/')[-1])

@login_required
def file_delete(request, pk):
    file = get_object_or_404(UserFile, pk=pk, owner=request.user)
    if request.method == 'POST':
        file.file.delete(save=False)
        file.delete()
        return redirect('file_list')
    return render(request, 'storage/file_confirm_delete.html', {'file': file})
"""
urls_code = """
from django.urls import path
from django.contrib.auth import views as auth_views
from . import views
urlpatterns = [
    path('', views.file_list, name='file_list'),
    path('upload/', views.upload, name='upload'),
    path('file/<int:pk>/', views.file_detail, name='file_detail'),
    path('file/<int:pk>/download/', views.file_download, name='file_download'),
    path('file/<int:pk>/delete/', views.file_delete, name='file_delete'),
    path('register/', views.register, name='register'),
    path('login/', auth_views.LoginView.as_view(template_name='storage/login.html'), name='login'),
    path('logout/', auth_views.LogoutView.as_view(), name='logout'),
]
"""
main_urls = """
from django.contrib import admin
from django.urls import path, include
from django.conf import settings
from django.conf.urls.static import static
urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('storage.urls')),
]
if settings.DEBUG:
    urlpatterns += static(settings.MEDIA_URL, document_root=settings.MEDIA_ROOT)
"""
with open('/content/config/storage/views.py', 'w') as f: f.write(views_code)
with open('/content/config/storage/urls.py', 'w') as f: f.write(urls_code)
with open('/content/config/config/urls.py', 'w') as f: f.write(main_urls)

# ==========================================
# 6. ШАБЛОНЫ И СТАТИКА
# ==========================================
os.makedirs('/content/config/storage/templates/storage', exist_ok=True)
os.makedirs('/content/config/storage/static/storage/css', exist_ok=True)

base = """{% load static %}<!DOCTYPE html><html lang="ru"><head><meta charset="UTF-8"><title>{% block title %}Облако{% endblock %}</title><link rel="stylesheet" href="{% static 'storage/css/style.css' %}"></head><body><header><a href="{% url 'file_list' %}">☁️ Моё облако</a><nav>{% if user.is_authenticated %}<a href="{% url 'upload' %}">Загрузить</a><span>Привет, {{ user.username }}</span><form action="{% url 'logout' %}" method="post" style="display:inline;">{% csrf_token %}<button type="submit">Выйти</button></form>{% else %}<a href="{% url 'login' %}">Войти</a><a href="{% url 'register' %}">Регистрация</a>{% endif %}</nav></header><main>{% block content %}{% endblock %}</main></body></html>"""
upload = """{% extends 'storage/base.html' %}{% block content %}<h1>Загрузить файл</h1><form method="post" enctype="multipart/form-data">{% csrf_token %}{{ form.as_p }}<button type="submit">Загрузить</button></form>{% endblock %}"""
flist = """{% extends 'storage/base.html' %}{% block content %}<h1>Мои файлы</h1><form method="get"><input type="text" name="q" placeholder="Поиск..." value="{{ query|default:'' }}"><select name="type"><option value="">Все типы</option><option value="image" {% if current_type == 'image' %}selected{% endif %}>Изображения</option><option value="video" {% if current_type == 'video' %}selected{% endif %}>Видео</option><option value="document" {% if current_type == 'document' %}selected{% endif %}>Документы</option></select><button type="submit">Фильтр</button></form><div class="file-grid">{% for file in files %}<div class="file-card">{% if file.is_image %}<img src="{{ file.file.url }}" style="max-width:100px;">{% elif file.is_video %}<video src="{{ file.file.url }}" controls style="max-width:150px;"></video>{% else %}<div>📄</div>{% endif %}<h3><a href="{{ file.get_absolute_url }}">{{ file.title|default:file.file.name }}</a></h3><p>{{ file.size_display }}</p><a href="{% url 'file_download' file.pk %}">Скачать</a> | <a href="{% url 'file_delete' file.pk %}">Удалить</a></div>{% empty %}<p>Файлов нет. <a href="{% url 'upload' %}">Загрузить</a></p>{% endfor %}</div>{% endblock %}"""
fdetail = """{% extends 'storage/base.html' %}{% block content %}<h1>{{ file.title|default:file.file.name }}</h1>{% if file.is_image %}<img src="{{ file.file.url }}" style="max-width:100%;">{% elif file.is_video %}<video src="{{ file.file.url }}" controls style="max-width:100%;"></video>{% else %}<p>Предпросмотр недоступен.</p>{% endif %}<ul><li>Тип: {{ file.get_file_type_display }}</li><li>Размер: {{ file.size_display }}</li><li>Публичный: {{ file.is_public|yesno:"да,нет" }}</li></ul><a href="{% url 'file_download' file.pk %}">Скачать</a> | <a href="{% url 'file_list' %}">← Назад</a>{% endblock %}"""
fdelete = """{% extends 'storage/base.html' %}{% block content %}<h1>Удалить файл?</h1><p>Вы уверены, что хотите удалить {{ file.title|default:file.file.name }}?</p><form method="post">{% csrf_token %}<button type="submit">Да, удалить</button></form><a href="{% url 'file_list' %}">Отмена</a>{% endblock %}"""
reg = """{% extends 'storage/base.html' %}{% block content %}<h1>Регистрация</h1><form method="post">{% csrf_token %}{{ form.as_p }}<button type="submit">Зарегистрироваться</button></form>{% endblock %}"""
login = """{% extends 'storage/base.html' %}{% block content %}<h1>Вход</h1><form method="post">{% csrf_token %}{{ form.as_p }}<button type="submit">Войти</button></form>{% endblock %}"""

for name, content in [('base.html', base), ('upload.html', upload), ('file_list.html', flist), ('file_detail.html', fdetail), ('file_confirm_delete.html', fdelete), ('register.html', reg), ('login.html', login)]:
    with open(f'/content/config/storage/templates/storage/{name}', 'w') as f: f.write(content)

with open('/content/config/storage/static/storage/css/style.css', 'w') as f:
    f.write("body { font-family: sans-serif; padding: 20px; background: #f9f9f9; } header { display: flex; justify-content: space-between; background: #333; color: white; padding: 10px; } header a { color: white; text-decoration: none; margin-right: 10px; } .file-grid { display: flex; flex-wrap: wrap; gap: 20px; } .file-card { border: 1px solid #ccc; padding: 10px; background: white; width: 200px; text-align: center; } img, video { border-radius: 5px; }")

# ==========================================
# 7. МИГРАЦИИ И ЗАПУСК
# ==========================================
!python manage.py makemigrations
!python manage.py migrate

os.environ['DJANGO_SUPERUSER_PASSWORD'] = 'admin12345'
!python manage.py createsuperuser --noinput --username admin --email admin@example.com

os.system('pkill -f runserver')
time.sleep(1)

def run_django():
    os.system('python manage.py runserver 0.0.0.0:8000')

threading.Thread(target=run_django).start()
time.sleep(5)

print("\n✅ ПРОЕКТ ГОТОВ! ВАША ССЫЛКА:")
print(eval_js("google.colab.kernel.proxyPort(8000)"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 20.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [django]
/content
/content/config
Migrations for 'storage':
  storage/migrations/0001_initial.py
    + Create model UserFile
Operations to perform:
  Apply all migrations: admin, auth, contenttypes, sessions, storage
Running migrations:
  Applying contenttypes.0001_initial... OK
  Applying auth.0001_initial... OK
  Applying admin.0001_initial... OK
  Applying admin.0002_logentry_remove_auto_add... OK
  Applying admin.0003_logentry_add_action_flag_choices... OK
  Applying contenttypes.0002_remove_content_type_name... OK
  Applying auth.0002_alter_permission_name_max_length... OK
  Applying auth.0003_alter_user_email_max_length... OK
  Applying auth.0004_alter_user_username_opts... OK
  Applying auth.0005_alter_user_last_login_null... OK
  Applying auth.0006_require_contenttypes_0002... OK
  Applying auth.0007_alter_validators_add_error_messages... OK
  Applying auth.